# 형성 에너지 실습

**Formation Energy**

물질의 에너지를 기준 원소 상태의 에너지와 비교한 값.

소재 분야에서 이해하기: 음수여도 다른 화합물로 분해되는 것이 더 유리할 수 있다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [Materials Project 용어집](https://docs.materialsproject.org/frequently-asked-questions/glossary-of-terms)

## 1. 형성 에너지 계산

총 에너지에서 기준 원소 상태의 에너지를 빼서 구합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

import pandas as pd

# 원소 기준 상태의 원자당 에너지 (가상 값, eV/atom)
reference = {'A': -3.20, 'B': -4.60, 'O': -4.90}

compounds = [
    {'formula': 'AB',    'counts': {'A': 1, 'B': 1},         'total_energy_eV': -8.55},
    {'formula': 'A2B',   'counts': {'A': 2, 'B': 1},         'total_energy_eV': -11.90},
    {'formula': 'AB2',   'counts': {'A': 1, 'B': 2},         'total_energy_eV': -12.95},
    {'formula': 'ABO3',  'counts': {'A': 1, 'B': 1, 'O': 3}, 'total_energy_eV': -25.30},
    {'formula': 'AB(un)','counts': {'A': 1, 'B': 1},         'total_energy_eV': -7.50},
]

def formation_energy(entry):
    atoms = sum(entry['counts'].values())
    reference_energy = sum(count * reference[element] for element, count in entry['counts'].items())
    return (entry['total_energy_eV'] - reference_energy) / atoms

table = pd.DataFrame([{'formula': entry['formula'],
                       'atoms': sum(entry['counts'].values()),
                       'E_total (eV)': entry['total_energy_eV'],
                       'E_form (eV/atom)': round(formation_energy(entry), 4)} for entry in compounds])
print(table)

In [ ]:
plt.bar(table['formula'], table['E_form (eV/atom)'])
plt.axhline(0, color='k', lw=1)
plt.ylabel('formation energy (eV/atom)'); plt.xticks(rotation=15); plt.show()
print('음수는 기준 원소 상태보다 안정하다는 뜻입니다.')
print('다만 음수여도 다른 화합물로 분해되는 것이 더 유리할 수 있습니다 -> 에너지 어보브 헐 노트북 참고')

## 2. 기준 상태를 잘못 고르면

In [ ]:
for shift in (0.0, 0.2, 0.5):
    wrong = dict(reference); wrong['B'] = reference['B'] + shift
    def formation_wrong(entry, table=wrong):
        atoms = sum(entry['counts'].values())
        return (entry['total_energy_eV'] - sum(c * table[e] for e, c in entry['counts'].items())) / atoms
    values = [formation_wrong(entry) for entry in compounds[:3]]
    print('B 기준 에너지 %+.1f eV 오차 -> 형성 에너지 %s' % (shift, np.round(values, 3)))
print('\n형성 에너지는 기준 상태 선택에 직접 의존합니다. 서로 다른 데이터베이스 값을 섞을 때 주의해야 합니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#formation-energy)을 여세요.